# Phase 2 — Train Transformer + VRM dự đoán Cluster-SID

Notebook này train `EncoderDecoderRetrievalModel` với VRM head trong `train_decoder.py`.

Luồng dữ liệu:

```text
prev_items → product_index → cluster SID sequence → Transformer → next cluster SID
                                                         ↓
                                              VRM head (InfoNCE) → rerank items trong cluster
```

Notebook dùng `semantic_ids.parquet` từ notebook 03 và `global_product_embeddings.f16.npy` từ notebook 02 (cho VRM). Codebook multi-resolution `1024 × 256 × 64`. Train 200k iterations.

Loss = SID cross-entropy (3 tầng) + VRM InfoNCE (in-batch negatives). Metric SID-level trên W&B.

## Trước khi chạy

1. Bật GPU trong Kaggle Notebook Settings.
2. Add Input là output `preprocessed` của notebook 01.
3. Add Input là output `vmarket_rqvae` của notebook 03.
4. Add Input là output `embeddings` của notebook 02 (cần cho VRM).
5. Tạo Kaggle Secret `GITHUB_TOKEN` có quyền đọc repository.
6. Tạo Kaggle Secret `WANDB_API_KEY`.
7. Notebook tự tìm session, Semantic ID, và embedding trong `/kaggle/input`; các cấu hình train còn lại lấy từ Gin.

## 0. Cấu hình

In [ ]:
from pathlib import Path

SESSION_ROOT = None
SEMANTIC_ID_ROOT = None
EMBEDDING_ROOT = None

GITHUB_REPOSITORY_URL = "https://github.com/nam-htran/VSF-MiniApp-Ecommerce.git"
GITHUB_BRANCH = "main"
REPOSITORY_ROOT = Path("/kaggle/working/vsf-miniapp-ecommerce-source")
AUTO_INSTALL_DEPENDENCIES = True

print("Configuration loaded.")

## 1. Cài dependency và kiểm tra GPU

In [ ]:
import importlib.metadata as metadata
import subprocess
import sys

from packaging.version import Version


requirements = {
    "gin-config": "0.5.0",
    "accelerate": "1.0.0",
    "einops": "0.8.0",
    "transformers": "4.46.0",
    "wandb": "0.19.0",
    "pyarrow": "16.0.0",
}
packages_to_install = []
for distribution, minimum_version in requirements.items():
    try:
        installed_version = metadata.version(distribution)
    except metadata.PackageNotFoundError:
        installed_version = None
    if installed_version is None or Version(installed_version) < Version(minimum_version):
        packages_to_install.append(f"{distribution}>={minimum_version}")

if AUTO_INSTALL_DEPENDENCIES and packages_to_install:
    print("Installing:", packages_to_install)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *packages_to_install])

import torch

if Version(torch.__version__.split("+")[0]) < Version("2.5.0"):
    raise RuntimeError(f"PyTorch >= 2.5.0 is required, found {torch.__version__}")

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Transformers:", metadata.version("transformers"))
print("CUDA available:", torch.cuda.is_available())
for index in range(torch.cuda.device_count()):
    properties = torch.cuda.get_device_properties(index)
    print(f"cuda:{index}: {properties.name}, {properties.total_memory / 2**30:.1f} GiB")
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before training the Transformer.")

## 2. Kết nối Weights & Biases

Đọc `WANDB_API_KEY` từ Kaggle Secret và đăng nhập W&B.

In [ ]:
import os


from kaggle_secrets import UserSecretsClient

os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")

## 3. Clone source từ GitHub

Đọc `GITHUB_TOKEN` từ Kaggle Secret và clone nhánh `main`.

In [ ]:
from kaggle_secrets import UserSecretsClient


github_token = UserSecretsClient().get_secret("GITHUB_TOKEN")
REPOSITORY_ROOT = Path(REPOSITORY_ROOT).expanduser().resolve()

git_environment = {
    **os.environ,
    "GITHUB_TOKEN": github_token,
    "GIT_TERMINAL_PROMPT": "0",
}
credential_helper = "!f() { echo username=x-access-token; echo password=$GITHUB_TOKEN; }; f"
git = ["git", "-c", f"credential.helper={credential_helper}"]

if (REPOSITORY_ROOT / ".git").is_dir():
    subprocess.run(
        [*git, "-C", str(REPOSITORY_ROOT), "pull", "--ff-only", "origin", GITHUB_BRANCH],
        check=True,
        env=git_environment,
    )
elif REPOSITORY_ROOT.exists():
    raise FileExistsError(f"Clone target is not a Git repository: {REPOSITORY_ROOT}")
else:
    subprocess.run(
        [*git, "clone", "--depth", "1", "--branch", GITHUB_BRANCH, GITHUB_REPOSITORY_URL, str(REPOSITORY_ROOT)],
        check=True,
        env=git_environment,
    )
del github_token, git_environment

SOURCE_ROOT = REPOSITORY_ROOT / "ai-recommendation/src"
if not (SOURCE_ROOT / "train_decoder.py").is_file():
    raise FileNotFoundError(f"Transformer source not found: {SOURCE_ROOT}")
print("SOURCE_ROOT:", SOURCE_ROOT)

## 4. Tìm input và tạo Gin runtime

In [ ]:
import json


def is_session_root(path):
    path = Path(path)
    return (
        (path / "model_sessions_train.parquet").is_file()
        and (path / "model_sessions_validation.parquet").is_file()
    )


def locate_session_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_session_root(root):
            return root
        raise FileNotFoundError(f"Notebook 01 session artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/preprocessed"),
        cwd / "preprocessed",
        cwd.parent / "preprocessed",
    ]
    for candidate in candidates:
        if is_session_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for manifest_path in kaggle_input.glob("**/preprocessing_manifest.json"):
            if is_session_root(manifest_path.parent):
                return manifest_path.parent.resolve()
        for train_path in kaggle_input.glob("**/model_sessions_train.parquet"):
            if is_session_root(train_path.parent):
                return train_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 01 output was not found. Add it as a Kaggle Dataset or set SESSION_ROOT."
    )


def is_semantic_id_root(path):
    return (Path(path) / "semantic_ids.parquet").is_file()


def locate_semantic_id_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_semantic_id_root(root):
            return root
        raise FileNotFoundError(f"Notebook 03 artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        cwd / "ai-recommendation/output/rq-vae",
        cwd / "output/rq-vae",
        cwd.parent / "output/rq-vae",
        Path("/kaggle/working/rq-vae"),
        Path("/kaggle/working/vmarket_rqvae"),
        cwd / "vmarket_rqvae",
        cwd.parent / "vmarket_rqvae",
    ]
    for candidate in candidates:
        if is_semantic_id_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for metrics_path in kaggle_input.glob("**/semantic_id_metrics.json"):
            if is_semantic_id_root(metrics_path.parent):
                return metrics_path.parent.resolve()
        for semantic_ids_path in kaggle_input.glob("**/semantic_ids.parquet"):
            if is_semantic_id_root(semantic_ids_path.parent):
                return semantic_ids_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 03 output was not found. Add it as a Kaggle Dataset or set SEMANTIC_ID_ROOT."
    )


def is_embedding_root(path):
    path = Path(path)
    return (
        (path / "global_product_embeddings.f16.npy").is_file()
        and (path / "global_embedding_index.parquet").is_file()
    )


def locate_embedding_root(explicit=None):
    if explicit is not None:
        root = Path(explicit).expanduser().resolve()
        if is_embedding_root(root):
            return root
        raise FileNotFoundError(f"Notebook 02 embedding artifacts were not found in {root}")

    cwd = Path.cwd().resolve()
    candidates = [
        Path("/kaggle/working/embeddings"),
        cwd / "embeddings",
        cwd.parent / "embeddings",
    ]
    for candidate in candidates:
        if is_embedding_root(candidate):
            return candidate.resolve()

    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for embedding_path in kaggle_input.glob("**/global_product_embeddings.f16.npy"):
            if is_embedding_root(embedding_path.parent):
                return embedding_path.parent.resolve()
    raise FileNotFoundError(
        "Notebook 02 output was not found. Add it as a Kaggle Dataset or set EMBEDDING_ROOT."
    )


SESSION_ROOT = locate_session_root(SESSION_ROOT)
SEMANTIC_ID_ROOT = locate_semantic_id_root(SEMANTIC_ID_ROOT)
EMBEDDING_ROOT = locate_embedding_root(EMBEDDING_ROOT)
SEMANTIC_IDS_PATH = SEMANTIC_ID_ROOT / "semantic_ids.parquet"
CONTENT_EMBEDDINGS_PATH = EMBEDDING_ROOT / "global_product_embeddings.f16.npy"

BASE_CONFIG_PATH = SOURCE_ROOT / "configs/transformer_vmarket_mr.gin"
CONFIG_PATH = Path("/kaggle/working/transformer_vmarket_mr.gin")
config_lines = BASE_CONFIG_PATH.read_text(encoding="utf-8").splitlines()
bindings = {
    "train.session_folder=": SESSION_ROOT,
    "train.semantic_ids_path=": SEMANTIC_IDS_PATH,
    "train.content_embeddings_path=": CONTENT_EMBEDDINGS_PATH,
}
for binding, value in bindings.items():
    matching_lines = [index for index, line in enumerate(config_lines) if line.startswith(binding)]
    if len(matching_lines) != 1:
        raise ValueError(f"Expected one {binding} binding, found {len(matching_lines)}")
    config_lines[matching_lines[0]] = binding + json.dumps(str(value))
CONFIG_PATH.write_text("\n".join(config_lines) + "\n", encoding="utf-8")

print("Session root:", SESSION_ROOT)
print("Semantic IDs:", SEMANTIC_IDS_PATH)
print("Content embeddings:", CONTENT_EMBEDDINGS_PATH)
print("Gin config:", CONFIG_PATH)

## 5. Train Transformer

In [ ]:
command = [sys.executable, "train_decoder.py", str(CONFIG_PATH)]
print("Running:", " ".join(command))
subprocess.run(command, cwd=SOURCE_ROOT, check=True)

## 6. Kiểm tra output

In [ ]:
OUTPUT_ROOT = Path("/kaggle/working/transformer")

In [ ]:
checkpoints = list(OUTPUT_ROOT.glob("checkpoint_*.pt"))
checkpoints.sort(key=lambda path: int(path.stem.rsplit("_", 1)[1]))
if not checkpoints:
    raise FileNotFoundError("No Transformer checkpoint was written.")
output_size = sum(path.stat().st_size for path in OUTPUT_ROOT.rglob("*") if path.is_file())

print("Transformer checkpoint validation: PASSED")
print("Latest checkpoint:", checkpoints[-1])
print("Output size:", f"{output_size / 2**30:.2f} GiB")
print("OUTPUT_ROOT:", OUTPUT_ROOT)

## Kết quả của bước này

Notebook hoàn thành khi cell cuối báo `Transformer checkpoint validation: PASSED`. Metric SID-level và VRM loss được log trên W&B. Bước tiếp theo mở rộng cluster thành candidate item và dùng VRM rerank để đánh giá item-level.